# Graph retrieval over the doc-search corpus

GraphRAG adds a graph of entities and relations to vector search. A query first finds the chunks
closest to the question, then follows the graph from those chunks to others that share an entity
with them, and passes both sets to the model as context. Libraries such as
[postgres-graph-rag](https://github.com/h4gen/postgres-graph-rag) keep the graph in Postgres next to
the pgvector column and use an LLM to extract the entities.

This notebook builds such a graph over the 407 `doc_chunks` of the lab's doc-search app without an
LLM. The SQLAlchemy documentation marks every API name it mentions with a reStructuredText role,
such as ``:class:`.Session` `` or ``:meth:`.Session.commit` ``, so each chunk's entities can be read
from its source. The notebook then compares vector search alone with vector search followed by graph
expansion, on 14 questions whose relevant sections were labeled by hand.

Run `just db` and `just ingest` first.

In [1]:
# Parameters. Papermill overrides these with `-p NAME value`.
SERVER = "pg16"  # the server that holds doc_chunks
K = 5  # chunks taken by vector search, and chunks added by the graph

In [2]:
import json
import math
import pathlib
import re
import statistics
import sys
import time

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "app")]
load_dotenv(ROOT / ".env")

import httpx
import pandas as pd
import psycopg
from fastembed import TextEmbedding
from pgvector.psycopg import register_vector

import ingest
from settings import settings

DSN = settings.sqlalchemy_url(SERVER).replace("postgresql+psycopg://", "postgresql://")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DSN, autocommit=True)
register_vector(conn)


def q(sql: str, params=None) -> pd.DataFrame:
    """Run one statement and show what came back."""
    cur = conn.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


print(conn.execute("SELECT count(*) FROM doc_chunks").fetchone()[0], "chunks in doc_chunks")

/Users/deburky/Documents/python-ai-projects/postgres-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


407 chunks in doc_chunks


## 1. Entities from the documentation source

`app/ingest.py` turns each page into chunks with `chunk_page`, which calls `clean_inline` on every
paragraph and replaces each role with its plain name. Wrapping `clean_inline` records which API
names each cleaned paragraph contained. Running `chunk_page` again on the same pages then gives the
same chunks as `doc_chunks`, and a chunk's entities are those of the paragraphs it is made of.
Names are normalized the way `clean_inline` prints them, so `:meth:`.Session.commit`` and
`:meth:`_orm.Session.commit`` both become `Session.commit`.

In [3]:
API_ROLE = re.compile(r":(class|meth|func|attr|paramref|obj|data):`([^`]*)`")


def entity_name(inner: str) -> str:
    inner = inner.lstrip("~")
    target = ingest.ROLE_TARGET.match(inner)
    inner = target.group(1) if target else inner
    inner = re.sub(r"^(_\w+\.|\.|sqlalchemy(\.\w+)*\.)", "", inner)
    return inner.lstrip("~.").removesuffix("()")


roles_in: dict[str, set[str]] = {}  # cleaned paragraph -> API names in its source
clean_inline = ingest.clean_inline


def recording_clean_inline(text: str) -> str:
    out = clean_inline(text)
    roles_in[out] = {entity_name(m.group(2)) for m in API_ROLE.finditer(text)}
    return out


ingest.clean_inline = recording_clean_inline
raw = {
    path: httpx.get(ingest.RAW.format(ref=settings.docs_ref, path=path), timeout=30).text
    for path in ingest.PAGES
}
stored = {
    (source, ordinal): (chunk_id, text)
    for chunk_id, source, ordinal, text in conn.execute("SELECT id, source, ordinal, text FROM doc_chunks")
}
mentions: dict[int, set[str]] = {}
for path, rst in raw.items():
    for ordinal, (_, text) in enumerate(ingest.chunk_page(path, rst, settings.chunk_words)):
        chunk_id, stored_text = stored[(path, ordinal)]
        assert text == stored_text, (path, ordinal)
        mentions[chunk_id] = set().union(*[names for para, names in roles_in.items() if para and para in text])
ingest.clean_inline = clean_inline

print(f"{len(mentions)} chunks, all identical to doc_chunks")
print(f"{len({e for es in mentions.values() for e in es})} entities, {sum(map(len, mentions.values()))} mentions")
print(f"{sum(1 for es in mentions.values() if not es)} chunks mention no API name")

407 chunks, all identical to doc_chunks
440 entities, 1436 mentions
33 chunks mention no API name


## 2. Two tables for the graph

The graph needs one table of entities and one of mentions, the edges between a chunk and an entity.
An index on `entity_id` serves the step from an entity back to its chunks.

In [4]:
conn.execute("""
DROP TABLE IF EXISTS doc_mentions, doc_entities;
CREATE TABLE doc_entities (id serial PRIMARY KEY, name text UNIQUE NOT NULL);
CREATE TABLE doc_mentions (
    chunk_id int REFERENCES doc_chunks (id) ON DELETE CASCADE,
    entity_id int REFERENCES doc_entities (id) ON DELETE CASCADE,
    PRIMARY KEY (chunk_id, entity_id)
);
CREATE INDEX ON doc_mentions (entity_id);
""")
with conn.cursor() as cur:
    names = sorted({e for es in mentions.values() for e in es})
    cur.executemany("INSERT INTO doc_entities (name) VALUES (%s)", [(n,) for n in names])
    ids = dict(conn.execute("SELECT name, id FROM doc_entities").fetchall())
    cur.executemany(
        "INSERT INTO doc_mentions VALUES (%s, %s)",
        [(chunk_id, ids[e]) for chunk_id, es in mentions.items() for e in es],
    )
conn.execute("ANALYZE doc_entities, doc_mentions")
q("""
SELECT e.name, count(*) AS chunks
FROM doc_mentions m JOIN doc_entities e ON e.id = m.entity_id
GROUP BY e.name ORDER BY chunks DESC, e.name LIMIT 12
""")

,name,chunks
0,Session,104
1,Table,53
2,Engine,44
3,relationship,43
4,Connection,41
5,Column,33
6,mapped_column,29
7,Mapped,24
8,select,22
9,MetaData,19


## 3. Hubs

A few entities appear in a large share of the chunks. `Session` alone is mentioned in about a
quarter of them, so any chunk that mentions it is two hops away from all the others that do. Most
entities sit at the other end, in a single chunk, where they connect nothing.

In [5]:
q("""
WITH df AS (SELECT entity_id, count(*) AS n FROM doc_mentions GROUP BY entity_id),
hop2 AS (
    SELECT m1.chunk_id,
           count(DISTINCT m2.chunk_id) AS all_entities,
           count(DISTINCT m2.chunk_id) FILTER (WHERE df.n <= 20) AS without_hubs
    FROM doc_mentions m1
    JOIN doc_mentions m2 ON m2.entity_id = m1.entity_id AND m2.chunk_id <> m1.chunk_id
    JOIN df ON df.entity_id = m1.entity_id
    GROUP BY m1.chunk_id
)
SELECT (SELECT count(*) FROM df WHERE n = 1) AS entities_in_one_chunk,
       (SELECT count(*) FROM df WHERE n > 20) AS entities_in_over_20,
       round(avg(all_entities)) AS avg_2hop_neighbours,
       round(avg(without_hubs)) AS avg_2hop_without_hubs
FROM hop2
""")

,entities_in_one_chunk,entities_in_over_20,avg_2hop_neighbours,avg_2hop_without_hubs
0,242,9,61,11


## 4. Hops with a recursive CTE

A recursive CTE walks the graph to any depth. Treating chunks and entities as nodes of one graph,
each step goes from a chunk to its entities or from an entity to its chunks, so even depths reach
chunks and odd depths reach entities. Starting from the chunk on implicit IO in `AsyncSession`, the
fourth hop reaches most of the corpus. Dropping the nine entities that appear in more than 20 chunks
slows the growth down.

In [6]:
start = conn.execute("""
    SELECT c.id FROM doc_chunks c JOIN doc_mentions m ON m.chunk_id = c.id
    WHERE c.title = 'Preventing Implicit IO when Using AsyncSession'
    GROUP BY c.id ORDER BY count(*) DESC LIMIT 1
""").fetchone()[0]
HOPS = """
WITH RECURSIVE kept AS (
    SELECT entity_id FROM doc_mentions GROUP BY entity_id HAVING count(*) <= %(max_df)s
),
edges (src, dst) AS (
    SELECT 'c' || chunk_id, 'e' || entity_id FROM doc_mentions JOIN kept USING (entity_id)
    UNION ALL
    SELECT 'e' || entity_id, 'c' || chunk_id FROM doc_mentions JOIN kept USING (entity_id)
),
walk (node, depth) AS (
    SELECT 'c' || %(start)s::int, 0
    UNION
    SELECT e.dst, w.depth + 1
    FROM walk w JOIN edges e ON e.src = w.node
    WHERE w.depth < 4
),
first_seen AS (SELECT node, min(depth) AS depth FROM walk GROUP BY node)
SELECT depth, CASE WHEN depth %% 2 = 0 THEN 'chunks' ELSE 'entities' END AS kind, count(*) AS new_nodes
FROM first_seen GROUP BY depth ORDER BY depth
"""
every = q(HOPS, {"start": start, "max_df": 10**6})
no_hubs = q(HOPS, {"start": start, "max_df": 20}).rename(columns={"new_nodes": "without hubs"})
every.merge(no_hubs, on=["depth", "kind"], how="left")

,depth,kind,new_nodes,without hubs
0,0,chunks,1,1
1,1,entities,12,10
2,2,chunks,146,37
3,3,entities,171,47
4,4,chunks,188,72


## 5. Vector search, then graph expansion

Vector search takes the `K` chunks closest to the question. The expansion follows two hops from
those seed chunks, through their entities to other chunks, and ranks each new chunk by the
entities it shares with the seeds. Each shared entity counts by its inverse document frequency,
`ln(N / df)`, so a link through `Session` counts for little and a link through a rare name such as
`AsyncSession.run_sync` counts for a lot.

In [7]:
model = TextEmbedding(settings.embed_model)
titles = {i: (s, t) for i, s, t in conn.execute("SELECT id, source, title FROM doc_chunks")}

VECTOR = "SELECT id FROM doc_chunks ORDER BY embedding <=> %s LIMIT %s"

EXPAND = """
WITH seeds AS (SELECT unnest(%(seeds)s::int[]) AS id),
idf AS (
    SELECT entity_id, ln((SELECT count(*) FROM doc_chunks)::float / count(*)) AS w
    FROM doc_mentions GROUP BY entity_id
)
SELECT m2.chunk_id AS id, sum(CASE WHEN %(weighted)s THEN idf.w ELSE 1 END) AS score
FROM seeds
JOIN doc_mentions m1 ON m1.chunk_id = seeds.id
JOIN doc_mentions m2 ON m2.entity_id = m1.entity_id
JOIN idf ON idf.entity_id = m1.entity_id
WHERE m2.chunk_id NOT IN (SELECT id FROM seeds)
GROUP BY m2.chunk_id
ORDER BY score DESC, m2.chunk_id
LIMIT %(k)s
"""


def vector_search(question: str, k: int) -> list[int]:
    v = next(model.query_embed([question]))
    return [r[0] for r in conn.execute(VECTOR, (v, k))]


def expand(seeds: list[int], k: int, weighted: bool = True) -> list[int]:
    return [r[0] for r in conn.execute(EXPAND, {"seeds": seeds, "k": k, "weighted": weighted})]


question = "What does expire_on_commit do and when should I turn it off?"
seeds = vector_search(question, K)
added = expand(seeds, K)
pd.DataFrame(
    [("vector", *titles[i]) for i in seeds] + [("graph", *titles[i]) for i in added],
    columns=["found by", "page", "section"],
)

,found by,page,section
0,vector,tutorial/orm_data_manipulation.rst,Closing a Session
1,vector,orm/session_state_management.rst,When to Expire or Refresh
2,vector,orm/session_state_management.rst,When to Expire or Refresh
3,vector,orm/session_basics.rst,Committing
4,vector,orm/session_basics.rst,Committing
5,graph,orm/session_state_management.rst,Refreshing / Expiring
6,graph,orm/queryguide/relationships.rst,Using contains_eager() to load a custom-filtered collection result
7,graph,orm/extensions/asyncio.rst,Preventing Implicit IO when Using AsyncSession
8,graph,orm/session_basics.rst,Opening and Closing a Session
9,graph,tutorial/dbapi_transactions.rst,Executing with an ORM Session


## 6. Labeled questions

Each question lists the sections of the corpus that answer it, chosen by reading the section titles
and text. Many of them span two pages, such as lazy loading in the query guide and implicit IO in the
asyncio extension, which is the case graph expansion is meant for. Recall is the share of a
question's labeled sections that at least one retrieved chunk belongs to.

Every method returns ten chunks, except vector top 5, so that a gain cannot come from a larger
context alone:

- **vector 5** and **vector 10**: the nearest chunks by cosine distance
- **vector 5 + graph 5**: the five nearest, then five more from the expansion above
- **unweighted**: the same expansion with every shared entity counting 1
- **vector 5 + neighbours by distance**: the five nearest, then the five nearest among the seeds'
  two-hop neighbours, so that the graph only restricts which chunks can be added

In [8]:
QUESTIONS = [
 {
  "q": "Why does accessing a relationship raise MissingGreenlet with AsyncSession?",
  "relevant": [
   [
    "orm/extensions/asyncio.rst",
    "Preventing Implicit IO when Using AsyncSession"
   ],
   [
    "orm/queryguide/relationships.rst",
    "Lazy Loading"
   ],
   [
    "orm/queryguide/relationships.rst",
    "Select IN loading"
   ],
   [
    "orm/queryguide/relationships.rst",
    "Preventing unwanted lazy loads using raiseload"
   ],
   [
    "tutorial/orm_related_objects.rst",
    "Selectin Load"
   ],
   [
    "tutorial/orm_related_objects.rst",
    "Raiseload"
   ]
  ]
 },
 {
  "q": "How do I call synchronous code, such as a pandas function, from an async engine?",
  "relevant": [
   [
    "orm/extensions/asyncio.rst",
    "Running Synchronous Methods and Functions under asyncio"
   ],
   [
    "orm/extensions/asyncio.rst",
    "Synopsis Core"
   ]
  ]
 },
 {
  "q": "What does expire_on_commit do and when should I turn it off?",
  "relevant": [
   [
    "orm/session_basics.rst",
    "Committing"
   ],
   [
    "orm/session_basics.rst",
    "Expiring / Refreshing"
   ],
   [
    "orm/session_state_management.rst",
    "Refreshing / Expiring"
   ],
   [
    "orm/session_state_management.rst",
    "When to Expire or Refresh"
   ],
   [
    "orm/extensions/asyncio.rst",
    "Preventing Implicit IO when Using AsyncSession"
   ]
  ]
 },
 {
  "q": "How do I handle database disconnects in the connection pool?",
  "relevant": [
   [
    "core/pooling.rst",
    "Dealing with Disconnects"
   ],
   [
    "core/pooling.rst",
    "Disconnect Handling Pessimistic"
   ],
   [
    "core/pooling.rst",
    "Disconnect Handling Optimistic"
   ],
   [
    "core/pooling.rst",
    "Setting Pool Recycle"
   ],
   [
    "core/pooling.rst",
    "More on Invalidation"
   ]
  ]
 },
 {
  "q": "How do I set the transaction isolation level or use autocommit?",
  "relevant": [
   [
    "core/connections.rst",
    "Setting Transaction Isolation Levels including DBAPI Autocommit"
   ],
   [
    "core/connections.rst",
    "Setting Isolation Level or DBAPI Autocommit for a Connection"
   ],
   [
    "core/connections.rst",
    "Setting Isolation Level or DBAPI Autocommit for an Engine"
   ],
   [
    "core/connections.rst",
    "Understanding the DBAPI-Level Autocommit Isolation Level"
   ],
   [
    "core/connections.rst",
    "Maintaining Multiple Isolation Levels for a Single Engine"
   ]
  ]
 },
 {
  "q": "How do I insert many rows at once and get the new primary keys back?",
  "relevant": [
   [
    "tutorial/data_insert.rst",
    "INSERT...RETURNING"
   ],
   [
    "core/connections.rst",
    "\"Insert Many Values\" Behavior for INSERT statements"
   ],
   [
    "core/connections.rst",
    "Correlating RETURNING rows to parameter sets"
   ],
   [
    "tutorial/orm_data_manipulation.rst",
    "Bulk / Multi Row INSERT, upsert, UPDATE and DELETE"
   ],
   [
    "tutorial/orm_data_manipulation.rst",
    "Autogenerated primary key attributes"
   ]
  ]
 },
 {
  "q": "How do I define a many-to-many relationship with an association table?",
  "relevant": [
   [
    "orm/basic_relationships.rst",
    "Many To Many"
   ],
   [
    "orm/basic_relationships.rst",
    "Association Object"
   ],
   [
    "orm/basic_relationships.rst",
    "Using a late-evaluated form for the \"secondary\" argument of many-to-many"
   ],
   [
    "orm/basic_relationships.rst",
    "Deleting Rows from the Many to Many Table"
   ]
  ]
 },
 {
  "q": "Can one AsyncSession be shared between concurrent asyncio tasks?",
  "relevant": [
   [
    "orm/extensions/asyncio.rst",
    "Using AsyncSession with Concurrent Tasks"
   ],
   [
    "orm/session_basics.rst",
    "Is the Session thread-safe? Is AsyncSession safe to share in concurrent tasks?"
   ]
  ]
 },
 {
  "q": "How do I stream a large result set without loading every row into memory?",
  "relevant": [
   [
    "core/connections.rst",
    "Using Server Side Cursors (a.k.a. stream results)"
   ],
   [
    "core/connections.rst",
    "Streaming with a fixed buffer via yield_per"
   ],
   [
    "core/connections.rst",
    "Streaming with a dynamically growing buffer using stream_results"
   ]
  ]
 },
 {
  "q": "When should I use joinedload and when selectinload?",
  "relevant": [
   [
    "orm/queryguide/relationships.rst",
    "Joined Eager Loading"
   ],
   [
    "orm/queryguide/relationships.rst",
    "Select IN loading"
   ],
   [
    "orm/queryguide/relationships.rst",
    "What Kind of Loading to Use ?"
   ],
   [
    "orm/queryguide/relationships.rst",
    "The Zen of Joined Eager Loading"
   ],
   [
    "tutorial/orm_related_objects.rst",
    "Joined Load"
   ],
   [
    "tutorial/orm_related_objects.rst",
    "Selectin Load"
   ]
  ]
 },
 {
  "q": "How do I map a Python Enum to a database column?",
  "relevant": [
   [
    "orm/declarative_tables.rst",
    "Using Python Enum or pep-586 Literal types in the type map"
   ],
   [
    "orm/declarative_tables.rst",
    "Native Enums and Naming"
   ],
   [
    "orm/declarative_tables.rst",
    "Altering the Configuration of the Default Enum"
   ],
   [
    "orm/declarative_tables.rst",
    "Linking Specific enum.Enum or typing.Literal to other datatypes"
   ]
  ]
 },
 {
  "q": "How do I use an engine's connection pool with multiprocessing or os.fork?",
  "relevant": [
   [
    "core/pooling.rst",
    "Using Connection Pools with Multiprocessing or os.fork()"
   ],
   [
    "core/connections.rst",
    "Engine Disposal"
   ]
  ]
 },
 {
  "q": "How is committing different with a Connection and with a Session?",
  "relevant": [
   [
    "tutorial/dbapi_transactions.rst",
    "Committing Changes"
   ],
   [
    "tutorial/dbapi_transactions.rst",
    "Executing with an ORM Session"
   ],
   [
    "core/connections.rst",
    "Commit As You Go"
   ],
   [
    "core/connections.rst",
    "Begin Once"
   ],
   [
    "orm/session_basics.rst",
    "Committing"
   ]
  ]
 },
 {
  "q": "How does the Session send an UPDATE when I change an attribute on an object?",
  "relevant": [
   [
    "tutorial/orm_data_manipulation.rst",
    "Updating ORM Objects using the Unit of Work pattern"
   ],
   [
    "tutorial/orm_data_manipulation.rst",
    "Flushing"
   ],
   [
    "orm/session_basics.rst",
    "Flushing"
   ]
  ]
 }
]

In [9]:
NEIGHBOURS = """
SELECT id FROM doc_chunks
WHERE id IN (
    SELECT m2.chunk_id FROM doc_mentions m1 JOIN doc_mentions m2 USING (entity_id)
    WHERE m1.chunk_id = ANY(%s) AND m2.chunk_id <> ALL(%s)
)
ORDER BY embedding <=> %s
LIMIT %s
"""


def recall(ids: list[int], relevant: set[tuple[str, str]]) -> float:
    return len({titles[i] for i in ids} & relevant) / len(relevant)


rows, timings = [], {"vector 10": [], "vector 5 + graph 5": []}
for item in QUESTIONS:
    relevant = {tuple(r) for r in item["relevant"]}
    v = next(model.query_embed([item["q"]]))
    t0 = time.perf_counter()
    top10 = [r[0] for r in conn.execute(VECTOR, (v, 2 * K))]
    timings["vector 10"].append(time.perf_counter() - t0)
    seeds = top10[:K]
    t0 = time.perf_counter()
    graph = expand(seeds, K)
    timings["vector 5 + graph 5"].append(time.perf_counter() - t0)
    neighbours = [r[0] for r in conn.execute(NEIGHBOURS, (seeds, seeds, v, K))]
    rows.append({
        "question": item["q"],
        "labeled": len(relevant),
        "vector 5": recall(seeds, relevant),
        "vector 10": recall(top10, relevant),
        "vector 5 + graph 5": recall(seeds + graph, relevant),
        "unweighted": recall(seeds + expand(seeds, K, weighted=False), relevant),
        "vector 5 + neighbours by distance": recall(seeds + neighbours, relevant),
        "same as vector 10": set(neighbours) == set(top10[K:]),
    })
results = pd.DataFrame(rows)
results.round(2)

,question,labeled,vector 5,vector 10,vector 5 + graph 5,unweighted,vector 5 + neighbours by distance,same as vector 10
0,Why does accessing a relationship raise MissingGreenlet with Async...,6,0.17,0.17,0.17,0.33,0.17,False
1,"How do I call synchronous code, such as a pandas function, from an...",2,1.00,1.00,1.00,1.00,1.00,True
2,What does expire_on_commit do and when should I turn it off?,5,0.40,0.60,0.80,0.60,0.60,True
3,How do I handle database disconnects in the connection pool?,5,0.60,1.00,0.80,0.60,1.00,True
4,How do I set the transaction isolation level or use autocommit?,5,0.40,1.00,0.60,0.60,0.80,False
5,How do I insert many rows at once and get the new primary keys back?,5,0.80,0.80,0.80,0.80,1.00,False
6,How do I define a many-to-many relationship with an association ta...,4,0.75,1.00,1.00,1.00,1.00,False
7,Can one AsyncSession be shared between concurrent asyncio tasks?,2,1.00,1.00,1.00,1.00,1.00,True
8,How do I stream a large result set without loading every row into ...,3,1.00,1.00,1.00,1.00,1.00,False
9,When should I use joinedload and when selectinload?,6,0.50,0.83,0.50,0.50,0.83,True


In [10]:
methods = ["vector 5", "vector 10", "vector 5 + graph 5", "unweighted", "vector 5 + neighbours by distance"]
summary = results[methods].mean().round(3).to_frame("mean recall")
summary["questions won vs vector 10"] = [(results[m] > results["vector 10"]).sum() for m in methods]
summary["questions lost vs vector 10"] = [(results[m] < results["vector 10"]).sum() for m in methods]
print(f"neighbours by distance picked the same five chunks as vector 10 on {results['same as vector 10'].sum()} of {len(results)} questions")
print({k: f"{1000 * statistics.median(v):.1f} ms" for k, v in timings.items()}, "median per query, after the embedding")
summary

neighbours by distance picked the same five chunks as vector 10 on 9 of 14 questions
{'vector 10': '5.1 ms', 'vector 5 + graph 5': '3.8 ms'} median per query, after the embedding


,mean recall,questions won vs vector 10,questions lost vs vector 10
vector 5,0.632,0,6
vector 10,0.788,0,0
vector 5 + graph 5,0.721,1,4
unweighted,0.705,1,4
vector 5 + neighbours by distance,0.788,1,1


## 7. Data for the in-browser console

The book's graph console loads the same graph without the embeddings: the chunks' pages and
titles, the entities and the mentions, written here as one SQL file.

In [11]:
def lit(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"


chunk_rows = conn.execute("SELECT id, source, title FROM doc_chunks ORDER BY id").fetchall()
entity_rows = conn.execute("SELECT id, name FROM doc_entities ORDER BY id").fetchall()
mention_rows = conn.execute("SELECT chunk_id, entity_id FROM doc_mentions ORDER BY 1, 2").fetchall()
sql = [
    "-- written by notebooks/graph_rag_playground.ipynb from doc_chunks, doc_entities and doc_mentions",
    "CREATE TABLE chunks (id int PRIMARY KEY, page text, title text);",
    "CREATE TABLE entities (id int PRIMARY KEY, name text UNIQUE);",
    "CREATE TABLE mentions (chunk_id int REFERENCES chunks, entity_id int REFERENCES entities, PRIMARY KEY (chunk_id, entity_id));",
    "INSERT INTO chunks VALUES\n" + ",\n".join(f"({i}, {lit(s.removesuffix('.rst'))}, {lit(t)})" for i, s, t in chunk_rows) + ";",
    "INSERT INTO entities VALUES\n" + ",\n".join(f"({i}, {lit(n)})" for i, n in entity_rows) + ";",
    "INSERT INTO mentions VALUES\n" + ",\n".join(f"({c}, {e})" for c, e in mention_rows) + ";",
    "CREATE INDEX ON mentions (entity_id);",
    "ANALYZE;",
]
out = ROOT / "viz" / "public" / "sql" / "doc-graph.sql"
out.write_text("\n".join(sql) + "\n")
print(out.relative_to(ROOT), f"{out.stat().st_size / 1024:.0f} kB")

viz/public/sql/doc-graph.sql 56 kB
